In [ ]:
#langgraph agent needs state object-? serves as shared memrory to accumaltes info.abs

from langgraph.graph import StateGraph,END
from typing import TypedDict, Annotated
import operator


## Setting Up state
#state that flows thru our graph
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    next_action: str
    iterations: int


In [3]:
##Tools creation
def search_tool(query: str) -> str:
    responses = {
        "weather tokyo": "XYZ Tokyo, cloudy",
        "population japan": "125 million japan"
    }
    return responses.get(query.lower(), f"NA")

In [4]:
# reasoning node
def reasoning_node(state: AgentState):
    messages = state["messages"]
    iterations = state.get("iterations", 0)

    if iterations  == 0:
        return {
            "messages": ["Though: Now I need TOkyo z"],
            "next_action": "action", "iterations": iterations+1
        }
    elif iterations  == 1:
        return {
            "messages": ["Though: Now I need Jap pop"],
            "next_action": "action", "iterations": iterations+1
        }
    else:
        return {
            "messages": ["Though: I have enough info"],
            "next_action": "end", "iterations": iterations+1
        }


In [6]:
# action node
def action_node(state:AgentState):
    iterations = state["iterations"]

    query = "weather tokyo" if iterations == 1 else "population japan"
    result = search_tool(query)

    return {
        "messages": [f"Action: Searched for '{query}'",
        f"Observation: {result}"],
        "next_action": "reasoning"
        }
    
# decides next step
def route(state:AgentState):
    return state["next_action"]

In [7]:
# building and executing graph

# grpah build
workflow = StateGraph(AgentState)
workflow.add_node("reasoning", reasoning_node)
workflow.add_node("action", action_node)

#define edges
workflow.set_entry_point("reasoning")
workflow.add_conditional_edges("reasoning", route,{
    "action": "action",
    "end": END
}
)
workflow.add_edge("action","reasoning")

# compile and run
app = workflow.compile()

# Execute
result = app.invoke(
    {"messages": ["User: Tell me about Tokyo and Japan"],
     "iteraions": 0, "next_action": ""}
)

# printflow

print("\n == React Output ==")
for msg in result["messages"]:
    print(msg)


 == React Output ==
User: Tell me about Tokyo and Japan
Though: Now I need TOkyo z
Action: Searched for 'weather tokyo'
Observation: XYZ Tokyo, cloudy
Though: Now I need Jap pop
Action: Searched for 'population japan'
Observation: 125 million japan
Though: I have enough info
